# Label Studio on Google Colab via cloudflared

Notebook ini menyiapkan Label Studio di Google Colab dengan `cloudflared` quick tunnel dan local file serving ke `/content/yolo/data`.

Environment yang dipakai:

- `LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=true`
- `LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT=/content/yolo/data`
- `USE_ENFORCE_CSRF_CHECKS=false`

Catatan:

- Quick tunnel `trycloudflare.com` cocok untuk testing/dev.
- Kalau tunnel mati, jalankan ulang cell `Start cloudflared tunnel`.
- Folder data project diasumsikan berada di `/content/yolo/data`.


## 1. Clone repo ke Colab

Kalau repo sudah ada di `/content/yolo`, cell ini aman dijalankan lagi.

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = os.environ.get("YOLO_REPO_URL", "https://github.com/faprikaa/yolo.git")
REPO_DIR = Path("/content/yolo")

if REPO_DIR.exists():
    print(f"Repo already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/yolo

## 2. Install dependency Label Studio dan utility tunnel

In [ ]:
%cd /content/yolo
!python -m pip install --upgrade pip
!python -m pip install label-studio label-studio-sdk requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb

## 3. Siapkan folder data dan environment Label Studio

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/yolo")
DATA_DIR = PROJECT_DIR / "data"
CAPTURE_DIR = DATA_DIR / "captures"
EXPORT_DIR = DATA_DIR / "exports"
DATASET_DIR = DATA_DIR / "datasets"
RUNS_DIR = DATA_DIR / "runs"

for path in (DATA_DIR, CAPTURE_DIR, EXPORT_DIR, DATASET_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.environ["LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED"] = "true"
os.environ["LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT"] = "/content/yolo/data"
os.environ["USE_ENFORCE_CSRF_CHECKS"] = "false"

os.environ["CAPTURE_DIR"] = str(CAPTURE_DIR)
os.environ["EXPORT_DIR"] = str(EXPORT_DIR)
os.environ["DATASET_DIR"] = str(DATASET_DIR)
os.environ["YOLO_RUNS_DIR"] = str(RUNS_DIR)

print("LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED=", os.environ["LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED"])
print("LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT=", os.environ["LABEL_STUDIO_LOCAL_FILES_DOCUMENT_ROOT"])
print("USE_ENFORCE_CSRF_CHECKS=", os.environ["USE_ENFORCE_CSRF_CHECKS"])


## 4. Jalankan Label Studio di background

Label Studio dijalankan pada `0.0.0.0:8080` supaya bisa dipublish ke tunnel.

In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

LABEL_STUDIO_LOG = Path("/content/label-studio.log")
LABEL_STUDIO_PID = Path("/content/label-studio.pid")

if LABEL_STUDIO_PID.exists():
    try:
        old_pid = int(LABEL_STUDIO_PID.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

command = [
    "label-studio",
    "start",
    "--host",
    "0.0.0.0",
    "--port",
    "8080",
]

with LABEL_STUDIO_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd="/content/yolo",
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

LABEL_STUDIO_PID.write_text(str(process.pid), encoding="utf-8")
print(f"Label Studio PID: {process.pid}")
time.sleep(8)
!tail -n 40 /content/label-studio.log

## 5. Start cloudflared tunnel

Cell ini membuat quick tunnel dari `http://127.0.0.1:8080` ke URL publik `trycloudflare.com`.

In [ ]:
import os
import re
import signal
import subprocess
import time
from pathlib import Path

CLOUDFLARED_LOG = Path("/content/cloudflared.log")
CLOUDFLARED_PID = Path("/content/cloudflared.pid")

if CLOUDFLARED_PID.exists():
    try:
        old_pid = int(CLOUDFLARED_PID.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

with CLOUDFLARED_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

CLOUDFLARED_PID.write_text(str(process.pid), encoding="utf-8")
print(f"cloudflared PID: {process.pid}")

public_url = None
for _ in range(30):
    time.sleep(2)
    log_text = CLOUDFLARED_LOG.read_text(encoding="utf-8", errors="ignore")
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("Label Studio public URL:", public_url)
else:
    print("Tunnel URL belum terdeteksi. Cek log berikut:")
    print(CLOUDFLARED_LOG.read_text(encoding="utf-8", errors="ignore"))

## 6. Cek log jika perlu

In [ ]:
!tail -n 60 /content/label-studio.log
!echo "---"
!tail -n 60 /content/cloudflared.log

## 7. Stop service

Jalankan jika ingin mematikan Label Studio dan `cloudflared`.

In [ ]:
import os
import signal
from pathlib import Path

for pid_file in (Path('/content/cloudflared.pid'), Path('/content/label-studio.pid')):
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
            print(f"Stopped {pid_file.name}")
        except Exception as error:
            print(f"Failed to stop {pid_file.name}: {error}")
